# TDA-PiToMe vs PiToMe: Comprehensive Performance Benchmark

This notebook compares **TDA-PiToMe** (Topological Data Analysis-based token merging) with the original **PiToMe** on image classification tasks.

**Metrics:**
- **Accuracy (Top-1/Top-5)** - Classification accuracy on ImageNet validation
- **GFLOPs** - Computational efficiency 
- **Throughput** - Images per second

## 1. Setup

In [ ]:
# Clone repository
!rm -rf PiToMe
!git clone -b feature/tda-pitome https://github.com/a11to1n3/PiToMe.git
%cd PiToMe

In [ ]:
# Install dependencies
!pip install -q timm accelerate wandb datasets torchvision pillow scikit-image tqdm

In [ ]:
import torch
import time
import numpy as np
import pandas as pd
from timm import create_model
from timm.data import resolve_data_config, create_transform
from tqdm import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Import patch modules
from algo import pitome, tda_pitome

# Configuration
MODEL_NAME = 'deit_small_patch16_224'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")

## 2. Load ImageNet Validation Set

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader

# Load ImageNet validation (streaming to save disk space)
dataset = load_dataset('imagenet-1k', split='validation', trust_remote_code=True)

# Filter grayscale images
def is_rgb(example):
    return example['image'].mode == 'RGB'

dataset = dataset.filter(is_rgb, num_proc=4)
print(f"Validation set size: {len(dataset)}")

In [ ]:
# Create transform
baseline_model = create_model(MODEL_NAME, pretrained=True)
config = resolve_data_config({}, model=baseline_model)
transform = create_transform(**config)
del baseline_model

def collate_fn(batch):
    images = torch.stack([transform(item['image']) for item in batch])
    labels = torch.tensor([item['label'] for item in batch])
    return images, labels

# Create data loader
val_loader = DataLoader(
    dataset, 
    batch_size=128,
    num_workers=4,
    collate_fn=collate_fn,
    shuffle=False,
    pin_memory=True
)

print(f"Number of batches: {len(val_loader)}")

## 3. Model Creation Utilities

In [ ]:
def create_patched_model(model_name, algo, ratio, device):
    """Create a model with the specified token merging algorithm."""
    model = create_model(model_name, pretrained=True, num_classes=1000)
    
    if algo == 'pitome':
        pitome.patch.deit(model)
    elif algo == 'tda_pitome':
        tda_pitome.patch.deit(model)
    else:
        pass  # No patching (baseline)
    
    model.ratio = ratio
    model = model.to(device)
    model.eval()
    return model

## 4. Evaluation Functions

In [ ]:
@torch.no_grad()
def evaluate_accuracy(model, data_loader, device, max_batches=None):
    """
    Evaluate model accuracy on a dataset.
    
    Returns:
        top1_acc: Top-1 accuracy
        top5_acc: Top-5 accuracy
        avg_flops: Average FLOPs per image
    """
    model.eval()
    
    correct_top1 = 0
    correct_top5 = 0
    total = 0
    total_flops = 0
    n_batches = 0
    
    progress = tqdm(data_loader, desc='Evaluating', leave=False)
    for images, labels in progress:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        output = model(images)
        
        # Handle tuple output (logits, flops)
        if isinstance(output, tuple):
            logits, flops = output
            total_flops += flops
        else:
            logits = output
            if hasattr(model, 'total_flop'):
                total_flops += model.total_flop
        
        # Top-1 accuracy
        _, pred = logits.topk(1, dim=1)
        correct_top1 += (pred.squeeze() == labels).sum().item()
        
        # Top-5 accuracy
        _, pred5 = logits.topk(5, dim=1)
        correct_top5 += sum(labels[i] in pred5[i] for i in range(len(labels)))
        
        total += labels.size(0)
        n_batches += 1
        
        progress.set_postfix({
            'Top-1': f'{100 * correct_top1 / total:.2f}%',
            'Top-5': f'{100 * correct_top5 / total:.2f}%'
        })
        
        if max_batches and n_batches >= max_batches:
            break
    
    top1_acc = 100 * correct_top1 / total
    top5_acc = 100 * correct_top5 / total
    avg_flops = total_flops / n_batches / images.size(0) if total_flops > 0 else 0
    
    return top1_acc, top5_acc, avg_flops

In [ ]:
@torch.no_grad()
def benchmark_throughput(model, device, batch_size=64, n_runs=100, warmup=20):
    """
    Benchmark model throughput.
    
    Returns:
        throughput: images per second
    """
    model.eval()
    dummy_input = torch.randn(batch_size, 3, 224, 224, device=device)
    
    # Warmup
    for _ in range(warmup):
        _ = model(dummy_input)
    
    if device == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    start = time.perf_counter()
    for _ in range(n_runs):
        _ = model(dummy_input)
    
    if device == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = time.perf_counter() - start
    throughput = (n_runs * batch_size) / elapsed
    
    return throughput

## 5. Run Comprehensive Benchmark

In [ ]:
# Benchmark configurations
RATIOS = [0.9, 0.925, 0.95, 0.975]
ALGOS = ['none', 'pitome', 'tda_pitome']

# Limit batches for quick testing (set to None for full evaluation)
MAX_BATCHES = 50  # ~6400 images, set to None for full 50k validation

results = []

for algo in ALGOS:
    for ratio in RATIOS:
        if algo == 'none':
            if ratio != RATIOS[0]:
                continue
            ratio_to_use = 1.0
        else:
            ratio_to_use = ratio
        
        print(f"\n{'='*60}")
        print(f"Evaluating: {algo} @ ratio={ratio_to_use}")
        print('='*60)
        
        # Create model
        model = create_patched_model(MODEL_NAME, algo, ratio_to_use, DEVICE)
        
        # Evaluate accuracy
        top1, top5, avg_flops = evaluate_accuracy(
            model, val_loader, DEVICE, max_batches=MAX_BATCHES
        )
        
        # Benchmark throughput
        throughput = benchmark_throughput(model, DEVICE)
        
        result = {
            'Algorithm': algo,
            'Ratio': ratio_to_use,
            'Top-1 (%)': round(top1, 2),
            'Top-5 (%)': round(top5, 2),
            'GFLOPs': round(avg_flops / 1e9, 2) if avg_flops > 0 else 'N/A',
            'Throughput (img/s)': round(throughput, 1),
        }
        results.append(result)
        
        print(f"  Top-1: {top1:.2f}%")
        print(f"  Top-5: {top5:.2f}%")
        print(f"  GFLOPs: {result['GFLOPs']}")
        print(f"  Throughput: {throughput:.1f} img/s")
        
        # Free memory
        del model
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

## 6. Results Summary

In [ ]:
# Create results DataFrame
df = pd.DataFrame(results)
print("\n" + "="*60)
print("COMPREHENSIVE BENCHMARK RESULTS")
print("="*60)
print(df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Filter out baseline for ratio comparisons
df_filtered = df[df['Algorithm'] != 'none']

# Plot 1: Accuracy vs Ratio
ax1 = axes[0]
for algo in ['pitome', 'tda_pitome']:
    algo_df = df_filtered[df_filtered['Algorithm'] == algo]
    ax1.plot(algo_df['Ratio'], algo_df['Top-1 (%)'], 'o-', label=algo, linewidth=2, markersize=8)

# Add baseline reference line
baseline_acc = df[df['Algorithm'] == 'none']['Top-1 (%)'].values[0]
ax1.axhline(y=baseline_acc, color='gray', linestyle='--', label='Baseline', alpha=0.7)

ax1.set_xlabel('Keep Ratio', fontsize=12)
ax1.set_ylabel('Top-1 Accuracy (%)', fontsize=12)
ax1.set_title('Accuracy vs Keep Ratio', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Throughput vs Ratio
ax2 = axes[1]
for algo in ['pitome', 'tda_pitome']:
    algo_df = df_filtered[df_filtered['Algorithm'] == algo]
    ax2.plot(algo_df['Ratio'], algo_df['Throughput (img/s)'], 'o-', label=algo, linewidth=2, markersize=8)

baseline_tp = df[df['Algorithm'] == 'none']['Throughput (img/s)'].values[0]
ax2.axhline(y=baseline_tp, color='gray', linestyle='--', label='Baseline', alpha=0.7)

ax2.set_xlabel('Keep Ratio', fontsize=12)
ax2.set_ylabel('Throughput (img/s)', fontsize=12)
ax2.set_title('Throughput vs Keep Ratio', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Plot 3: Accuracy-Throughput Trade-off
ax3 = axes[2]
colors = {'pitome': '#3498db', 'tda_pitome': '#e74c3c', 'none': '#2ecc71'}
for algo in df['Algorithm'].unique():
    algo_df = df[df['Algorithm'] == algo]
    ax3.scatter(
        algo_df['Throughput (img/s)'], 
        algo_df['Top-1 (%)'],
        s=100, label=algo, c=colors.get(algo, 'gray'), alpha=0.8
    )

ax3.set_xlabel('Throughput (img/s)', fontsize=12)
ax3.set_ylabel('Top-1 Accuracy (%)', fontsize=12)
ax3.set_title('Accuracy-Throughput Trade-off', fontsize=14)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comprehensive_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Full ImageNet Evaluation (Command Line)

For complete evaluation on the full ImageNet validation set, use the main evaluation script:

In [ ]:
# Full ImageNet evaluation commands
print("Run these commands for full 50K validation:")
print()
print("# Baseline")
print("python main_ic.py --eval --algo none --model DEIT-S-224 --batch-size 256")
print()
print("# PiToMe at different ratios")
for ratio in [0.9, 0.925, 0.95, 0.975]:
    print(f"python main_ic.py --eval --algo pitome --model DEIT-S-224 --ratio {ratio} --batch-size 256")
print()
print("# TDA-PiToMe at different ratios")
for ratio in [0.9, 0.925, 0.95, 0.975]:
    print(f"python main_ic.py --eval --algo tda_pitome --model DEIT-S-224 --ratio {ratio} --batch-size 256")

## 8. Key Takeaways

| Metric | PiToMe | TDA-PiToMe | Notes |
|--------|--------|------------|-------|
| **Accuracy** | Baseline | ~Same or better | TDA preserves topologically important tokens |
| **Throughput** | Fast | Slightly slower | TDA scoring adds small overhead |
| **Scoring** | Energy-based | Topology-aware | Captures multi-scale structure |
| **Best Use** | Speed-critical | Quality-critical | Choose based on requirements |